# 🩺 DoCare AI - LLM Fine-Tuning with QLoRA (Google Colab T4 GPU)

Notebook ini dirancang untuk melakukan fine-tuning model LLM (Meta Llama 3.2 3B / 1B) menggunakan teknik **QLoRA (Parameter-Efficient Fine-Tuning)** pada dataset medis MedQuAD.

**Persyaratan Hardware**: Google Colab T4 GPU (Gratis, Runtime -> Change runtime type -> T4 GPU).

In [ ]:
# 1. Install Library Pendukung
!pip install -q -U transformers datasets trl peft bitsandbytes accelerate

In [ ]:
# 2. Import Library & Cek GPU
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import pandas as pd

print('GPU Tersedia:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'HANYA CPU (Harap ganti runtime ke T4 GPU!)')

In [ ]:
# 3. Model ID & Konfigurasi 4-bit Quantization
model_id = 'unsloth/Llama-3.2-3B-Instruct' # Base model ringan dan cepat

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# 4. Konfigurasi LoRA Adapter (Hanya melatih ~1% parameter)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. Format Dataset MedQuAD ke Format Chat DoCare AI
# Silakan unggah file cleaned_medquad.csv ke Google Colab files
df = pd.read_csv('cleaned_medquad.csv').head(2000) # Latih 2.000 sampel terbaik

def format_prompts(batch):
    formatted_texts = []
    for q, a in zip(batch['question'], batch['answer']):
        text = (
            f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
            f'Anda adalah DoCare AI, asisten pendamping kesehatan terpercaya.<|eot_id|>'
            f'<|start_header_id|>user<|end_header_id|>\n{q}<|eot_id|>'
            f'<|start_header_id|>assistant<|end_header_id|>\nTerima kasih telah berkonsultasi dengan DoCare AI. '
            f'{a} Semoga informasi ini membantu dan lekas sembuh.<|eot_id|>'
        )
        formatted_texts.append(text)
    return {'text': formatted_texts}

dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_prompts, batched=True)

In [ ]:
# 6. Eksekusi Training (SFTTrainer)
training_args = TrainingArguments(
    output_dir='./docare-llama-lora',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    fp16=True,
    optim='paged_adamw_8bit',
    save_strategy='steps',
    save_steps=50,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print('Memulai Fine-Tuning Llama 3.2 dengan QLoRA...')
trainer.train()
print('Fine-Tuning Selesai! Menyimpan LoRA Adapter...')
model.save_pretrained('./docare-llama-lora-final')
tokenizer.save_pretrained('./docare-llama-lora-final')